# 第 0 课 · Web 零基础前置（写给完全不懂网页的你）

> **内核提醒（每课都要看）**：本 notebook 必须在 conda 环境 **ai_learn** 中运行。
> 右上角内核应显示 `Python 3.10.21 ('ai_learn')`；打开后从上到下逐格运行（Shift+Enter）。
> 若报 `ModuleNotFoundError`，先检查内核是否选成了 ai_learn。

> 这一课几乎不写代码，只做一件事：让你彻底搞懂 **网页、浏览器、服务器、HTML、爬虫** 到底是什么。
> 你平时写的是控制台程序，我们会用你熟悉的东西（函数调用、文件、多叉树）来类比。一次只讲一个概念。

## 0.1 先建立大图景：互联网、客户端与服务器

- **互联网**：全世界无数台电脑用网线 / WiFi / 光纤连成的一张大网。
- 网上的电脑分两种角色：
  - **客户端（client）**：你自己的电脑、你的浏览器，负责“提要求、看结果”。
  - **服务器（server）**：存放网站内容、24 小时开机的电脑，负责“接要求、给内容”。

**客户端—服务器模型**，用餐厅类比：
- 你（客户端）看菜单、点菜（这叫**请求 request**）；
- 后厨（服务器）做好、端给你（这叫**响应 response**），端上来的通常是 HTML。

**用 C++ 类比**：这很像一次函数调用——你“调用”服务器上的某个功能、传参数，它“返回”结果。
区别在于这个函数在遥远的另一台机器上，你们靠网络来回传消息；**这种消息的格式规范就叫 HTTP**。

```
  你的电脑（客户端 / 浏览器）                     服务器（存着网站）
 ┌───────────────────────┐   ① 请求：给我首页     ┌──────────────────┐
 │      Chrome 浏览器     │ ───────────────────▶ │   网站服务器      │
 │                       │ ◀─────────────────── │ 24h开机，存着HTML │
 └───────────────────────┘   ② 响应：HTML 文本    └──────────────────┘
```

## 0.2 从输入网址到看到页面，发生了什么（全课最重要）

这是后面所有课都在反复拆解的主线，请先有个整体印象：

1. 你在地址栏输入 `www.xxx.com` 并回车。
2. **DNS 解析**：浏览器先问“`www.xxx.com` 这台服务器的 **IP 地址**是多少？”
   DNS 像通讯录 / 电话簿：域名好记，IP（如 `93.184.216.34`）才是真正的门牌号。
3. **建立连接**：浏览器按 IP 找到服务器，先建立一条网络连接（TCP，类比“先拨通电话”）。
4. **发送 HTTP 请求**：浏览器通过连接发一段规范的话，例如“我想用 **GET** 方法，请求 `/` 这个页面”。
5. **服务器返回 HTTP 响应**：服务器找到内容，把 **HTML 文本**打包发回来。
6. **浏览器渲染**：浏览器拿到的只是一堆 HTML 文本，它读这堆文本、“画”出你看到的漂亮页面。

```
输入网址 → DNS查IP → 建立连接 → 发HTTP请求 → 服务器返回HTML → 浏览器渲染成页面
```

> **关键认知（爬虫的伏笔）**：服务器返回的本质是**一段 HTML 文本**，漂亮页面是浏览器“画”出来的。
> **爬虫 = 写程序替你完成第 4、5 步拿到 HTML，但通常不做第 6 步渲染，直接从文本里抠数据。**
> 所以“网页上看到的”和“程序拿到的文本”可能不一样，第 11 课讲动态页面时你会亲眼见到。

## 0.3 HTML 是什么

**HTML** 是一种用“标签（tag）”描述网页结构的**纯文本**，服务器返回给浏览器的就是它。先看一个最小例子：

```html
<html>
  <body>
    <h1>我的第一个网页</h1>
    <p class="intro">这是一个<a href="https://example.com">链接</a>。</p>
  </body>
</html>
```

逐行看四个概念：
- **标签成对出现**：`<h1>` 开始、`</h1>` 结束，表示“这是什么内容”。`h1`=大标题，`p`=段落，`a`=链接。
- **嵌套**：标签里可以再放标签（`p` 里放了 `a`）。
- **属性（attribute）**：开始标签里可带额外信息。`a` 的 `href` 是链接地址；`p` 的 `class="intro"` 是类别名——**后面找数据主要就靠 class**。
- **文本（text）**：标签中间夹的文字，才是真正显示出来的内容。

## 0.4 为什么 HTML 天然是一棵树

因为标签层层嵌套：最外层的 `<html>` 像树根，里面的标签是子节点，再往里是孙节点。把上面的最小例子画出来：

```
<html>
 └─ <body>
     ├─ <h1> → 文本“我的第一个网页”
     └─ <p class="intro">
         ├─ 文本“这是一个”
         └─ <a href="..."> → 文本“链接”
```

- 这正是你熟悉的**多叉树**：每个节点有父节点、若干子节点。
- 后面“解析 HTML”，本质就是在这棵树上做**查找 / 遍历**，和你图论里遍历节点没有本质区别。
- 所以别被“网页”吓到，你真正要做的还是老本行：在一棵树上找节点。

## 0.5 爬虫到底在干嘛（一句话 + 对比）

```
浏览器： 请求 → 拿到HTML → 渲染成页面（画给人看）
爬虫：   请求 → 拿到HTML → 在HTML树上提取数据 → 存成 CSV / SQLite（给程序用）
```

- 爬虫比浏览器**少了“渲染给人看”这一步**，多了“按规则抠数据并保存”这一步。
- 你的优势非常明显：抠数据 = 在多叉树上定位节点，这是你竞赛里练了无数次的能力。

## 0.6 动手：认识 F12 开发者工具（请跟着做）

1. 打开 `https://books.toscrape.com/`，在页面空白处右键 → **“检查”（或按 F12）**。
2. 先认识两个最常用的面板：
   - **Elements（元素）**：就是这页的 **HTML 树**。点左上角的“鼠标箭头”图标，再点页面上任意内容，会自动跳到对应的 HTML 节点——你能亲眼确认“页面 = HTML 树”。
   - **Network（网络）**：打开它之后按 F5 刷新页面，会看到浏览器发出的**一条条请求**；点任意一条，能看到它的 **Request Headers（请求头）** 和 **Response（响应内容）**。这正是我们的程序要复现的东西。
3. 现在看不懂细节完全没关系，先混个眼熟，后面每一课都会回来用它。

## 0.7 随堂小练习（动手做，不用交）

1. 在 books.toscrape.com 按 F12，用箭头工具点一本书的标题，看 Elements 里高亮的是哪个标签、它外面套了哪些标签。
2. 切到 Network 再刷新，大致数一下这一页发了多少条请求。
3. 在 Elements 里找一个带 `class=` 的标签（例如 `<article class="product_pod">`），记住这个 class 的值，第 7 课会用它。

### 先看一眼：HTML 其实就是个文本文件

下面用 Python 打开本地夹具 HTML，让你直观看到“网页 = 一段字符串”。

In [1]:
from pathlib import Path
DATA_DIR = Path(r'C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\08_爬虫\data')
html_text = (DATA_DIR / 'books_index.html').read_text(encoding='utf-8')
print('类型：', type(html_text))      # <class 'str'>，就是字符串
print('字符数：', len(html_text))
print('前 180 个字符：')
print(html_text[:180])

类型： <class 'str'>
字符数： 2398
前 180 个字符：
<!DOCTYPE html>
<html lang="zh-CN">
<head>
    <meta charset="UTF-8">
    <title>在线书店 · 全部书籍（第 1 页）</title>
    <style>
        body { font-family: "Microsoft YaHei", sans-serif; m


## 来源与延伸

- MDN《Web 入门》（Mozilla 官方，最权威的零基础前端教程）：https://developer.mozilla.org/zh-CN/docs/Learn/Getting_started_with_the_web
- MDN 客户端—服务器模型：https://developer.mozilla.org/zh-CN/docs/Learn/Server-side/First_steps/Client-Server_overview
- DNS 工作原理（Cloudflare 学习中心）：https://www.cloudflare.com/zh-cn/learning/dns/what-is-dns/
- HTML 入门（MDN）：https://developer.mozilla.org/zh-CN/docs/Learn/HTML/Introduction_to_HTML/Getting_started
- 浏览器渲染原理（推荐，英文）：https://developer.mozilla.org/en-US/docs/Web/Performance/How_browsers_work